## Import & Config & Path settings

In [1]:
# %%
import sys
import os
import numpy as np
import pandas as pd
from pathlib import Path
import warnings

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# --- 1. Project Path Setup ---
def find_project_root(name='Bambino', start=None):
    if start is None: start = os.getcwd()
    parent = start
    while True:
        if os.path.basename(parent) == name: return parent
        if os.path.dirname(parent) == parent: return None
        parent = os.path.dirname(parent)

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT: 
    sys.path.append(PROJECT_ROOT)
    print(f"Project Root found: {PROJECT_ROOT}")
else:
    print("Warning: Project root 'Bambino' not found.")

# --- 2. Imports ---
import config_ad as cfg
import data_utils_ad
import moment_ad_utils
import vis_utils
from config import settings as global_settings

# %% --- 3. Load Data & Create AD Splits ---
print("\n>>> Step 1: Loading Data & Creating AD Splits")
print(f"Dataset Type: {cfg.TRAIN_SPLIT_RATIO*100}% Normal for Training")

# This calls the updated function that handles BoaOpenFaceDataset
train_loader, test_loader = data_utils_ad.get_ad_dataloaders()

# Quick Verification of shapes
sample_x, sample_y, sample_meta = next(iter(train_loader))
print(f"\nBatch Verification:")
print(f"  X Shape: {sample_x.shape} (Expected: [B, 38, 512])")
print(f"  Y Shape: {sample_y.shape} (Should be all {cfg.NORMAL_CLASS})")

Project Root found: /home/phd2/Scrivania/CorsoRepo/Bambino

>>> Step 1: Loading Data & Creating AD Splits
Dataset Type: 75.0% Normal for Training
Loading all datasets...
   -> Loading: training_set.pt...
   -> Loading: validation_set.pt...
   -> Loading: test_set.pt...
Total Samples: 896
  - Normal (Class 1): 719
  - Anomaly (Class 0): 177

AD Data Preparation Complete:
  -> Train Loader: 539 samples (Clean Normals)
  -> Test Loader:  357 samples (180 Normals, 177 Anomalies)

Batch Verification:
  X Shape: torch.Size([16, 38, 512]) (Expected: [B, 38, 512])
  Y Shape: torch.Size([16]) (Should be all 1)


In [2]:
sample_y

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

## Moment initialization

In [3]:
# %% --- 2. Initialize Model ---
detector = moment_ad_utils.MomentAnomalyDetector()

Loading MOMENT (Reconstruction) on cuda...


## Zero-Shot

In [4]:
# print some specs of the test loader
print(f"\nTest Loader Specs:")
print(f"  Number of Batches: {len(test_loader)}")
print(f"  Batch Size: {test_loader.batch_size}")
print(f"  Total Samples: {len(test_loader.dataset)}")
print(f"  Normal Samples: {sum(test_loader.dataset.labels == cfg.NORMAL_CLASS)}")
print(f"  Anomalous Samples: {sum(test_loader.dataset.labels != cfg.NORMAL_CLASS)}")


Test Loader Specs:
  Number of Batches: 23
  Batch Size: 16
  Total Samples: 357
  Normal Samples: 180
  Anomalous Samples: 177


In [5]:
# %% --- 3. Zero-Shot Evaluation ---
print("\n>>> Step 2: Zero-Shot Evaluation (Before Fine-tuning)")
mse_zs, lbl_zs, vis_zs = detector.predict(test_loader)
print("Zero-Shot evaluation completed.")


>>> Step 2: Zero-Shot Evaluation (Before Fine-tuning)
Running Inference...


100%|██████████| 23/23 [00:31<00:00,  1.36s/it]

Zero-Shot evaluation completed.


In [8]:
# print some specs of mse_zs and lbl_zs and vis_zs
print(f"\nZero-Shot Evaluation Specs:")
print(f"  MSE Shape: {mse_zs.shape} (Expected: [{len(test_loader.dataset)}])")
print(f"  Labels Shape: {lbl_zs.shape} (Expected: [{len(test_loader.dataset)}])")
print(f"  Visualizations Count: {len(vis_zs)} (Expected: [{len(test_loader.dataset)}])")

# print the first 5 mse_zs and lbl_zs and vis_zs
print(f"\nZero-Shot Evaluation Samples:")
print(f"  First n MSE: {mse_zs[300:310]}")
print(f"  First n Labels: {lbl_zs[300:310]}")



Zero-Shot Evaluation Specs:
  MSE Shape: (357,) (Expected: [357])
  Labels Shape: (357,) (Expected: [357])
  Visualizations Count: 3 (Expected: [357])

Zero-Shot Evaluation Samples:
  First n MSE: [0.06046042 0.18616645 0.32290533 0.06951371 0.07457159 0.05915649
 0.10122063 0.10236461 0.05061524 0.1361361 ]
  First n Labels: [0 0 0 0 0 0 0 0 0 0]


In [9]:
# Visualize Zero-Shot
result_dir_zero_shot = os.path.join(cfg.BASE_DIR, "zero_shot_results")
os.makedirs(result_dir_zero_shot, exist_ok=True)

vis_utils.plot_boxplot_errors(mse_zs, lbl_zs, result_dir_zero_shot, prefix="ZeroShot")
vis_utils.plot_reconstruction_examples(vis_zs, result_dir_zero_shot, prefix="ZeroShot")

print("Zero-Shot visualization completed.")
print(f"Results saved to {result_dir_zero_shot}")


--- ZeroShot Stats ---
Stimulus Mean MSE: 0.0768 (std: 0.0675)
Control  Mean MSE: 0.0832 (std: 0.0716)
Zero-Shot visualization completed.
Results saved to /home/phd2/Scrivania/CorsoRepo/Bambino/_03_train/moment_anomaly_detection/zero_shot_results


## Fine-Tuning

In [10]:
# %% --- 4. Fine-Tuning ---
print("\n>>> Step 3: Fine-Tuning on Normal Data (Stimulus Only)")
detector.fine_tune(train_loader)


>>> Step 3: Fine-Tuning on Normal Data (Stimulus Only)
Starting Fine-tuning...


Epoch 1/10: 100%|██████████| 34/34 [00:49<00:00,  1.44s/it]


Epoch 1 Loss: 0.190539


Epoch 2/10: 100%|██████████| 34/34 [00:50<00:00,  1.49s/it]


Epoch 2 Loss: 0.157015


...

Epoch 8 Loss: 0.103097


Epoch 9/10: 100%|██████████| 34/34 [00:50<00:00,  1.50s/it]


Epoch 9 Loss: 0.099713


Epoch 10/10: 100%|██████████| 34/34 [00:50<00:00,  1.50s/it]

Epoch 10 Loss: 0.097972


In [11]:
# %% --- 5. Fine-Tuned Evaluation ---
print("\n>>> Step 4: Fine-Tuned Evaluation")
mse_ft, lbl_ft, vis_ft = detector.predict(test_loader)
print("Fine-Tuned evaluation completed.")


>>> Step 4: Fine-Tuned Evaluation
Running Inference...


100%|██████████| 23/23 [00:31<00:00,  1.38s/it]

Fine-Tuned evaluation completed.


In [13]:
# print some specs of mse_zs and lbl_zs and vis_zs
print(f"\nFine-Tuned Evaluation Specs:")
print(f"  MSE Shape: {mse_ft.shape} (Expected: [{len(test_loader.dataset)}])")
print(f"  Labels Shape: {lbl_ft.shape} (Expected: [{len(test_loader.dataset)}])")
print(f"  Visualizations Count: {len(vis_ft)} (Expected: [{len(test_loader.dataset)}])")

# print the first 5 mse_zs and lbl_zs for class 0 and 5 for class 1
print(f"\nFine-Tuned Evaluation Samples:")
class_0_indices = np.where(lbl_ft == cfg.NORMAL_CLASS)[0][:5]
class_1_indices = np.where(lbl_ft != cfg.NORMAL_CLASS)[0][:5]
print(f"  Class {cfg.NORMAL_CLASS} MSE: {mse_ft[class_0_indices]}")
print(f"  Class {cfg.NORMAL_CLASS} Labels: {lbl_ft[class_0_indices]}")
print(f"  Class {cfg.ANOMALY_CLASS} MSE: {mse_ft[class_1_indices]}")
print(f"  Class {cfg.ANOMALY_CLASS} Labels: {lbl_ft[class_1_indices]}")


Fine-Tuned Evaluation Specs:
  MSE Shape: (357,) (Expected: [357])
  Labels Shape: (357,) (Expected: [357])
  Visualizations Count: 3 (Expected: [357])

Fine-Tuned Evaluation Samples:
  Class 1 MSE: [0.02549439 0.01598174 0.0166206  0.01735723 0.03867928]
  Class 1 Labels: [1 1 1 1 1]
  Class 0 MSE: [0.02259055 0.12235717 0.01168772 0.03375734 0.00462711]
  Class 0 Labels: [0 0 0 0 0]


In [14]:
# Visualize Fine-Tuned
result_dir_fine_tuned = os.path.join(cfg.BASE_DIR, "fine_tuned_results")
os.makedirs(result_dir_fine_tuned, exist_ok=True)

vis_utils.plot_boxplot_errors(mse_ft, lbl_ft, result_dir_fine_tuned, prefix="FineTuned")
vis_utils.plot_reconstruction_examples(vis_ft, result_dir_fine_tuned, prefix="FineTuned")

print("Fine-Tuned visualization completed.")
print(f"Results saved to {result_dir_fine_tuned}")


--- FineTuned Stats ---
Stimulus Mean MSE: 0.0298 (std: 0.0254)
Control  Mean MSE: 0.0327 (std: 0.0285)
Fine-Tuned visualization completed.
Results saved to /home/phd2/Scrivania/CorsoRepo/Bambino/_03_train/moment_anomaly_detection/fine_tuned_results
